## Embedding Model Comparison

Compare `all-MiniLM-L6-v2` (22M) vs `microsoft/harrier-oss-v1-270m` (270M) on retrieval quality.

**Setup:** Anglo American AR2019 chunks → ChromaDB → 8 ground truth queries.

**Metrics:** Recall@5, MRR (dense-only and hybrid with BM25 + RRF).

In [14]:
import os
if os.name == "nt":
    os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import torch  # must load before numpy-dependent libs on Windows

In [15]:
import sys, json, time, re
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
import chromadb

# Ensure repo root is on path for utils imports
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

print(f"Repo root: {REPO_ROOT}")
print(f"torch {torch.__version__}")

Repo root: c:\Users\elimo\problem-set-2-elimossmarks11
torch 2.5.1+cpu


### 1. Load chunks (Anglo AR2019 only)

In [16]:
# Try per-PDF parquet first, fall back to company-level
per_pdf = Path(REPO_ROOT) / "data" / "interim" / "anglo" / "AR2019" / "chunks.parquet"
company_level = Path(REPO_ROOT) / "data" / "interim" / "anglo" / "chunks.parquet"

if per_pdf.exists():
    df = pd.read_parquet(per_pdf)
    print(f"Loaded per-PDF parquet: {len(df)} chunks")
elif company_level.exists():
    df = pd.read_parquet(company_level)
    df = df[df["source"].str.contains("AR2019")]
    print(f"Filtered from company parquet: {len(df)} chunks")
else:
    raise FileNotFoundError(
        f"No chunks found. Run 'python chunking.py chunk --company anglo' first.\n"
        f"Checked: {per_pdf}\n         {company_level}"
    )

chunks = df.to_dict(orient="records")
print(f"Columns: {list(df.columns)}")
df.head(3)

Loaded per-PDF parquet: 991 chunks
Columns: ['id', 'text', 'strategy', 'source', 'pages', 'element_types', 'page_num', 'is_table']


,id,text,strategy,source,pages,element_types,page_num,is_table
0,elem_0000,HEADER (H2): STRATEGIC REPORT CHAIRMAN’S STATE...,element_type,anglo/AR2019,"[6, 7]","[Footer, Header, NarrativeText]",NaN,None
1,elem_0001,HEADER (H2): STRATEGIC REPORT CHAIRMAN’S STATE...,element_type,anglo/AR2019,"[6, 7]","[Footer, Header, NarrativeText]",NaN,None
2,elem_0002,HEADER (H2): STRATEGIC REPORT CHAIRMAN’S STATE...,element_type,anglo/AR2019,"[6, 7]","[Footer, Header, NarrativeText]",NaN,None


### 2. Load ground truth

In [17]:
gt_path = Path(REPO_ROOT) / "ground_truth.json"
ground_truth = json.loads(gt_path.read_text(encoding="utf-8"))

# Filter to AR2019 queries only (source matches)
gt_queries = [q for q in ground_truth if "AR2019" in q.get("source", "")]
print(f"{len(gt_queries)} queries targeting AR2019:")
for q in gt_queries:
    print(f"  - {q['query'][:80]}")

6 queries targeting AR2019:
  - What are Anglo American's energy use targets for 2020?
  - What are Anglo American's GHG emissions targets for 2020?
  - What were Anglo American's Scope 1 Emissions in CO2e in 2019?
  - Does Anglo American account for downstream emissions?
  - What organisational boundary does Anglo American use for emissions reporting?
  - What was the underlying EBIT for De Beers in 2019


### 3. Define models

In [18]:
MODELS = {
    "MiniLM-L6-v2": {
        "name": "sentence-transformers/all-MiniLM-L6-v2",
        "query_prompt": None,
        "prompt_name": None,
    },
    "harrier-270m (no prompt)": {
        "name": "microsoft/harrier-oss-v1-270m",
        "query_prompt": None,
        "prompt_name": None,
    },
    "harrier-270m (prompted)": {
        "name": "microsoft/harrier-oss-v1-270m",
        "query_prompt": None,
        "prompt_name": "web_search_query",
    },
}

### 4. Snippet-matching relevance function

Ground truth has `snippet` text, not chunk IDs. We find relevant chunks by checking which chunks contain the snippet (or a high overlap).

In [19]:
def find_relevant_chunk_ids(
    snippet: str, chunks: list[dict], overlap_threshold: float = 0.5
) -> set[str]:
    """Find chunk IDs whose text contains most of the ground truth snippet."""
    snippet_words = set(snippet.lower().split())
    relevant = set()
    for c in chunks:
        chunk_words = set(c["text"].lower().split())
        overlap = len(snippet_words & chunk_words) / len(snippet_words)
        if overlap >= overlap_threshold:
            relevant.add(c["id"])
    return relevant

# Pre-compute relevant IDs for each query
for q in gt_queries:
    q["relevant_ids"] = find_relevant_chunk_ids(q["snippet"], chunks)
    status = f"{len(q['relevant_ids'])} chunks" if q["relevant_ids"] else "NO MATCH"
    print(f"  [{status}] {q['query'][:70]}")

# Warn about unmatched queries
unmatched = [q for q in gt_queries if not q["relevant_ids"]]
if unmatched:
    print(f"\nWARNING: {len(unmatched)} queries have no matching chunks — check snippet overlap.")

  [3 chunks] What are Anglo American's energy use targets for 2020?
  [2 chunks] What are Anglo American's GHG emissions targets for 2020?
  [2 chunks] What were Anglo American's Scope 1 Emissions in CO2e in 2019?
  [8 chunks] Does Anglo American account for downstream emissions?
  [2 chunks] What organisational boundary does Anglo American use for emissions rep
  [1 chunks] What was the underlying EBIT for De Beers in 2019


### 5. Embed & evaluate (dense + hybrid)

In [20]:
K = 5
RRF_K = 60  # standard RRF constant (Cormack et al. 2009)
# retrieve more candidates than K so RRF fusion has headroom
N_CANDIDATES_MULTIPLIER = 2


def tokenize_bm25(text: str) -> list[str]:
    """Simple whitespace + punctuation tokenizer for BM25."""
    return re.findall(r"\w+", text.lower())


def bm25_rank(
    query: str, chunks: list[dict], k: int = K, b: float = 0.75, k1: float = 1.5
) -> list[str]:
    """Return top-k chunk IDs ranked by BM25."""
    q_tokens = tokenize_bm25(query)
    n = len(chunks)

    doc_tokens = [tokenize_bm25(c["text"]) for c in chunks]
    avg_dl = sum(len(dt) for dt in doc_tokens) / n if n else 1
    df = Counter()
    for dt in doc_tokens:
        for token in set(dt):
            df[token] += 1

    scores = []
    for i, dt in enumerate(doc_tokens):
        tf = Counter(dt)
        dl = len(dt)
        score = 0.0
        for qt in q_tokens:
            if qt not in df:
                continue
            idf = np.log((n - df[qt] + 0.5) / (df[qt] + 0.5) + 1)
            tf_norm = (tf[qt] * (k1 + 1)) / (tf[qt] + k1 * (1 - b + b * dl / avg_dl))
            score += idf * tf_norm
        scores.append((score, chunks[i]["id"]))

    scores.sort(key=lambda x: x[0], reverse=True)
    return [cid for _, cid in scores[:k]]


def rrf_fuse(ranked_lists: list[list[str]], k: int = K) -> list[str]:
    """Reciprocal Rank Fusion over multiple ranked ID lists."""
    scores: dict[str, float] = {}
    for ranked in ranked_lists:
        for rank, cid in enumerate(ranked, 1):
            scores[cid] = scores.get(cid, 0.0) + 1.0 / (RRF_K + rank)
    return sorted(scores, key=scores.get, reverse=True)[:k]


def compute_metrics(retrieved_ids: list[str], relevant_ids: set[str]) -> tuple[float, float]:
    """Compute recall@K and MRR for a single query."""
    hits = len(relevant_ids & set(retrieved_ids))
    recall = hits / len(relevant_ids) if relevant_ids else 0.0
    mrr = 0.0
    for rank, rid in enumerate(retrieved_ids, 1):
        if rid in relevant_ids:
            mrr = 1.0 / rank
            break
    return recall, mrr


def evaluate_model(
    model_label: str,
    model_cfg: dict,
    chunks: list[dict],
    queries: list[dict],
    k: int = K,
) -> pd.DataFrame:
    """Embed chunks, evaluate both dense and hybrid (BM25 + RRF) retrieval."""
    print(f"\n{'='*50}")
    print(f"  {model_label}: {model_cfg['name']}")
    print(f"{'='*50}")

    # Load model
    t0 = time.time()
    model = SentenceTransformer(model_cfg["name"])
    print(f"  Model loaded in {time.time()-t0:.1f}s")

    # Embed documents
    t0 = time.time()
    doc_texts = [c["text"] for c in chunks]
    doc_embeddings = model.encode(doc_texts, normalize_embeddings=True, show_progress_bar=True)
    embed_time = time.time() - t0
    print(f"  Embedded {len(chunks)} chunks in {embed_time:.1f}s ({embed_time/len(chunks)*1000:.0f}ms/chunk)")
    print(f"  Embedding dim: {doc_embeddings.shape[1]}")

    # Build ephemeral ChromaDB collection
    client = chromadb.Client()
    col_name = f"eval{abs(hash(model_label)) % 10**8}"
    collection = client.create_collection(name=col_name, metadata={"hnsw:space": "cosine"})
    collection.add(
        ids=[c["id"] for c in chunks],
        embeddings=doc_embeddings.tolist(),
        documents=doc_texts,
    )

    n_candidates = k * N_CANDIDATES_MULTIPLIER
    rows = []
    for entry in queries:
        query = entry["query"]
        relevant = entry["relevant_ids"]
        if not relevant:
            continue

        # Encode query
        encode_kwargs = {"normalize_embeddings": True}
        if model_cfg.get("prompt_name"):
            encode_kwargs["prompt_name"] = model_cfg["prompt_name"]
        q_vec = model.encode([query], **encode_kwargs)[0]

        # Dense retrieval
        results = collection.query(query_embeddings=[q_vec.tolist()], n_results=n_candidates)
        dense_ranked = results["ids"][0]
        recall, mrr = compute_metrics(dense_ranked[:k], relevant)
        rows.append({"model": model_label, "method": "dense", "query": query[:60], "recall@5": recall, "mrr": mrr})

        # Hybrid retrieval (dense + BM25 via RRF)
        bm25_ranked = bm25_rank(query, chunks, k=n_candidates)
        fused = rrf_fuse([dense_ranked, bm25_ranked], k=k)
        recall, mrr = compute_metrics(fused, relevant)
        rows.append({"model": model_label, "method": "hybrid", "query": query[:60], "recall@5": recall, "mrr": mrr})

    # Clean up
    client.delete_collection(col_name)
    del model
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

    return pd.DataFrame(rows)

In [21]:
all_results = pd.concat(
    [evaluate_model(label, cfg, chunks, gt_queries) for label, cfg in MODELS.items()],
    ignore_index=True,
)
all_results


  MiniLM-L6-v2: sentence-transformers/all-MiniLM-L6-v2
  Model loaded in 1.7s


Batches:   0%|          | 0/31 [00:00<?, ?it/s]

  Embedded 991 chunks in 19.4s (20ms/chunk)
  Embedding dim: 384

  harrier-270m (no prompt): microsoft/harrier-oss-v1-270m
  Model loaded in 4.6s


Batches:   0%|          | 0/31 [00:00<?, ?it/s]

  Embedded 991 chunks in 400.5s (404ms/chunk)
  Embedding dim: 640

  harrier-270m (prompted): microsoft/harrier-oss-v1-270m
  Model loaded in 5.0s


Batches:   0%|          | 0/31 [00:00<?, ?it/s]

  Embedded 991 chunks in 399.4s (403ms/chunk)
  Embedding dim: 640


,model,method,query,recall@5,mrr
0,MiniLM-L6-v2,dense,What are Anglo American's energy use targets f...,0.666667,0.333333
1,MiniLM-L6-v2,hybrid,What are Anglo American's energy use targets f...,0.666667,0.500000
2,MiniLM-L6-v2,dense,What are Anglo American's GHG emissions target...,0.500000,0.250000
3,MiniLM-L6-v2,hybrid,What are Anglo American's GHG emissions target...,0.500000,0.333333
4,MiniLM-L6-v2,dense,What were Anglo American's Scope 1 Emissions i...,0.500000,0.500000
5,MiniLM-L6-v2,hybrid,What were Anglo American's Scope 1 Emissions i...,0.500000,0.500000
6,MiniLM-L6-v2,dense,Does Anglo American account for downstream emi...,0.125000,0.200000
7,MiniLM-L6-v2,hybrid,Does Anglo American account for downstream emi...,0.125000,0.500000
8,MiniLM-L6-v2,dense,What organisational boundary does Anglo Americ...,0.500000,0.500000
9,MiniLM-L6-v2,hybrid,What organisational boundary does Anglo Americ...,0.500000,1.000000


### 6. Summary table

In [22]:
summary = (
    all_results.groupby(["model", "method"])
    .agg({"recall@5": "mean", "mrr": "mean"})
    .round(3)
    .sort_values(["recall@5", "mrr"], ascending=False)
    .reset_index()
)

print("\n" + "=" * 55)
print("  EMBEDDING MODEL COMPARISON — SUMMARY")
print("=" * 55)
summary


  EMBEDDING MODEL COMPARISON — SUMMARY


,model,method,recall@5,mrr
0,MiniLM-L6-v2,hybrid,0.549,0.506
1,harrier-270m (no prompt),hybrid,0.417,0.333
2,harrier-270m (prompted),dense,0.417,0.292
3,harrier-270m (prompted),hybrid,0.389,0.500
4,MiniLM-L6-v2,dense,0.382,0.297
5,harrier-270m (no prompt),dense,0.250,0.250
